# Klasifikasi triase dengan XGBoost

Alur: CSV lokal → pemeriksaan kualitas dan sebaran label → perbandingan model → tuning → evaluasi pada data uji.

Jalankan sel dari atas ke bawah menggunakan lingkungan Python proyek. Notebook membaca CSV lokal dan menampilkan hasil sebagai teks. Data, grafik, dan model tidak diekspor.

## 1. Muat dataset lokal

Dataset TriageGeist berisi delapan input: suhu (°C), SpO₂ (%), laju napas (napas/menit), denyut jantung (bpm), tekanan sistolik dan diastolik (mmHg), GCS, dan skor nyeri (0–10). Target `triage_acuity` menyatakan level triase 1–5. Ringkasan di bawah menghitung jumlah dan proporsi tiap level dari CSV yang dipakai.

Acuan: branch `test` commit `4a86fd7`, `model/xgboost_triase.py:39,92` dan notebook `model/Triage_XGBoost (1).ipynb`.

In [1]:
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

KOLOM_INPUT = [
    "temperature_c", "spo2", "respiratory_rate", "heart_rate",
    "systolic_bp", "diastolic_bp", "gcs_total", "pain_score",
]
TARGET = "triage_acuity"
PEMETAAN = {1: 0, 2: 0, 3: 1, 4: 2, 5: 2}
NAMA_KELAS = {0: "Resusitasi", 1: "Darurat", 2: "Non-Darurat"}
KELAS = list(NAMA_KELAS)
SEED = 42

folder = Path.cwd()
root = next((path for path in (folder, *folder.parents) if (path / "model" / "xgboost_triase.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Jalankan notebook dari folder proyek TriaGo atau subfoldernya.")

lokasi_csv = [
    root / "model" / "train.csv",
    Path.home() / ".cache" / "kagglehub" / "competitions" / "triagegeist" / "train.csv",
]
sumber_csv = next((path for path in lokasi_csv if path.is_file()), None)
if sumber_csv is None:
    raise FileNotFoundError("CSV lokal belum tersedia. Lokasi yang diperiksa: " + ", ".join(map(str, lokasi_csv)))

data_mentah = pd.read_csv(sumber_csv, dtype={"patient_id": "string"})
kolom_hilang = [kolom for kolom in [*KOLOM_INPUT, TARGET] if kolom not in data_mentah]
if kolom_hilang:
    raise ValueError(f"Kolom wajib hilang: {', '.join(kolom_hilang)}")

def distribusi_label(label, urutan):
    jumlah = label.value_counts().reindex(urutan, fill_value=0)
    return pd.DataFrame({"Jumlah": jumlah, "Proporsi (%)": (jumlah / len(label) * 100).round(2)})

print(f"Sumber CSV: {sumber_csv}")
print(f"Jumlah baris: {len(data_mentah):,}")
print("Distribusi level asli:")
print(distribusi_label(pd.to_numeric(data_mentah[TARGET], errors="coerce"), list(PEMETAAN)).to_string())

c:\Users\Adyty\Documents\Farid ITS\TriaGo\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sumber CSV: C:\Users\Adyty\.cache\kagglehub\competitions\triagegeist\train.csv
Jumlah baris: 80,000
Distribusi level asli:
               Jumlah  Proporsi (%)
triage_acuity                      
1                3222          4.03
2               13439         16.80
3               28921         36.15
4               23020         28.78
5               11398         14.25


## 2. Periksa kualitas data dan sebaran kelas

Level 1–2 digabung menjadi Resusitasi, level 3 menjadi Darurat, dan level 4–5 menjadi Non-Darurat. Perbandingan proporsi memakai data setelah duplikat identik dikeluarkan sebagai titik awal. Baris yang keluar karena input kosong dapat mengubah proporsi kelas.

Akurasi merangkum semua prediksi. Macro-F1 memberi bobot yang sama pada setiap kelas, sementara recall Resusitasi menunjukkan bagian kasus Resusitasi yang berhasil dikenali.

In [2]:
def siapkan_data(data_mentah):
    data = data_mentah.drop_duplicates()
    print(f"Duplikat identik dibuang: {len(data_mentah) - len(data):,}")

    if "patient_id" in data:
        id_peserta = data["patient_id"].astype("string").str.strip()
        id_kosong = id_peserta.isna() | id_peserta.eq("")
        id_berulang = id_peserta[~id_kosong].duplicated().sum()
        print(f"patient_id kosong: {id_kosong.sum():,}, kemunculan berulang: {id_berulang:,}")
        if id_kosong.any() or id_berulang:
            raise ValueError("Perbaiki patient_id kosong atau berulang sebelum membagi data.")
    else:
        print("Kolom patient_id tidak tersedia. Pemeriksaan per peserta tidak dapat dilakukan.")

    input_asli = data[KOLOM_INPUT]
    input_angka = input_asli.apply(pd.to_numeric, errors="coerce")
    nonnumerik = (input_asli.notna() & input_angka.isna()).sum()
    tak_hingga = input_angka.isin([np.inf, -np.inf]).sum()
    for nama, jumlah in [("nonnumerik", nonnumerik), ("tak hingga", tak_hingga)]:
        if jumlah.any():
            raise ValueError(f"Input {nama} per kolom: {jumlah[jumlah > 0].to_dict()}")

    input_angka["pain_score"] = input_angka["pain_score"].replace(-1, np.nan)
    nyeri = input_angka["pain_score"]
    nyeri_di_luar_batas = nyeri.notna() & ~nyeri.between(0, 10)
    if nyeri_di_luar_batas.any():
        raise ValueError(f"pain_score di luar 0–10: {nyeri_di_luar_batas.sum():,} baris.")

    level = pd.to_numeric(data[TARGET], errors="coerce")
    label_invalid = ~level.isin(PEMETAAN)
    if label_invalid.any():
        raise ValueError(f"{TARGET} kosong atau di luar level 1–5: {label_invalid.sum():,} baris.")

    print("Input kosong per kolom setelah menangani pain_score = -1:")
    print(input_angka.isna().sum().to_string())
    bp_terbalik = input_angka["systolic_bp"] <= input_angka["diastolic_bp"]
    print(f"Sistolik ≤ diastolik (diagnostik): {bp_terbalik.sum():,} baris.")

    label_sebelum = level.map(PEMETAAN).astype("int8")
    input_bersih = input_angka.dropna().astype("float32")
    label_bersih = label_sebelum.loc[input_bersih.index]
    print(f"Baris dibuang karena input kosong: {len(input_angka) - len(input_bersih):,}")
    if set(label_bersih) != set(KELAS):
        raise ValueError("Data bersih harus memiliki ketiga kelas triase.")
    return input_bersih, label_bersih, label_sebelum

input_bersih, label_bersih, label_sebelum = siapkan_data(data_mentah)
sebelum = distribusi_label(label_sebelum, KELAS)
sesudah = distribusi_label(label_bersih, KELAS)
ringkasan = pd.concat({"Sebelum": sebelum, "Sesudah": sesudah}, axis=1).rename(index=NAMA_KELAS)
print(f"Jumlah baris siap dimodelkan: {len(input_bersih):,}")
print(ringkasan.to_string())
print("\nGrafik batang teks (satu # kira-kira 2 poin persentase):")
for kode, nama in NAMA_KELAS.items():
    print(nama)
    for tahap, tabel in [("Sebelum", sebelum), ("Sesudah", sesudah)]:
        proporsi = tabel.loc[kode, "Proporsi (%)"]
        print(f"  {tahap:7} {'#' * round(proporsi / 2)} {proporsi:.2f}%")

Duplikat identik dibuang: 0
patient_id kosong: 0, kemunculan berulang: 0
Input kosong per kolom setelah menangani pain_score = -1:
temperature_c         574
spo2                    0
respiratory_rate     3067
heart_rate              0
systolic_bp          4146
diastolic_bp         4146
gcs_total               0
pain_score          11156
Sistolik ≤ diastolik (diagnostik): 2,105 baris.
Baris dibuang karena input kosong: 17,171
Jumlah baris siap dimodelkan: 62,829
              Sebelum              Sesudah             
               Jumlah Proporsi (%)  Jumlah Proporsi (%)
triage_acuity                                          
Resusitasi      16661        20.83   16661        26.52
Darurat         28921        36.15   23729        37.77
Non-Darurat     34418        43.02   22439        35.71

Grafik batang teks (satu # kira-kira 2 poin persentase):
Resusitasi
  Sebelum ########## 20.83%
  Sesudah ############# 26.52%
Darurat
  Sebelum ################## 36.15%
  Sesudah ################

## 3. Pisahkan data latih, validasi, dan uji

Stratifikasi menjaga proporsi tiap kelas mendekati proporsi data bersih. Model belajar dari set latih. Set validasi dipakai untuk perbandingan dan tuning. Set uji disisihkan sampai model akhir ditentukan.

In [3]:
input_sementara, input_uji, label_sementara, label_uji = train_test_split(
    input_bersih, label_bersih, test_size=0.10, stratify=label_bersih, random_state=SEED,
)
input_latih, input_validasi, label_latih, label_validasi = train_test_split(
    input_sementara, label_sementara, test_size=1 / 9, stratify=label_sementara, random_state=SEED,
)

assert input_latih.index.intersection(input_validasi.index).empty
assert input_latih.index.intersection(input_uji.index).empty
assert input_validasi.index.intersection(input_uji.index).empty

for nama, label in [("Latih", label_latih), ("Validasi", label_validasi), ("Uji", label_uji)]:
    print(f"\n{nama}: {len(label):,} baris")
    print(distribusi_label(label, KELAS).rename(index=NAMA_KELAS).to_string())


Latih: 50,263 baris
               Jumlah  Proporsi (%)
triage_acuity                      
Resusitasi      13329         26.52
Darurat         18983         37.77
Non-Darurat     17951         35.71

Validasi: 6,283 baris
               Jumlah  Proporsi (%)
triage_acuity                      
Resusitasi       1666         26.52
Darurat          2373         37.77
Non-Darurat      2244         35.72

Uji: 6,283 baris
               Jumlah  Proporsi (%)
triage_acuity                      
Resusitasi       1666         26.52
Darurat          2373         37.77
Non-Darurat      2244         35.72


## 4. Bentuk sebelas fitur

Shock index membagi denyut jantung dengan tekanan sistolik. Tekanan nadi adalah selisih sistolik dan diastolik. MAP adalah perkiraan tekanan arteri rata-rata: diastolik ditambah sepertiga tekanan nadi.

In [4]:
KOLOM_FITUR = [*KOLOM_INPUT, "shock_index", "pulse_pressure", "MAP"]

def bentuk_fitur(input_vital):
    fitur = input_vital.copy()
    sistolik = fitur["systolic_bp"]
    diastolik = fitur["diastolic_bp"]
    fitur["shock_index"] = fitur["heart_rate"] / sistolik.clip(lower=1)
    fitur["pulse_pressure"] = sistolik - diastolik
    fitur["MAP"] = diastolik + fitur["pulse_pressure"] / 3
    return fitur

fitur_latih = bentuk_fitur(input_latih)
fitur_validasi = bentuk_fitur(input_validasi)
fitur_uji = bentuk_fitur(input_uji)
for fitur in (fitur_latih, fitur_validasi, fitur_uji):
    assert list(fitur.columns) == KOLOM_FITUR

contoh = bentuk_fitur(pd.DataFrame({"heart_rate": [72], "systolic_bp": [120], "diastolic_bp": [80]}))
assert np.isclose(contoh["pulse_pressure"].iloc[0], 40)
assert np.isclose(contoh["MAP"].iloc[0], 80 + 40 / 3)
print("Urutan fitur:", ", ".join(KOLOM_FITUR))
print(f"Contoh 120/80 mmHg: tekanan nadi {contoh['pulse_pressure'].iloc[0]:.0f}, MAP {contoh['MAP'].iloc[0]:.2f} mmHg.")

Urutan fitur: temperature_c, spo2, respiratory_rate, heart_rate, systolic_bp, diastolic_bp, gcs_total, pain_score, shock_index, pulse_pressure, MAP
Contoh 120/80 mmHg: tekanan nadi 40, MAP 93.33 mmHg.


## 5. Bandingkan lima model pada validasi

Baseline adalah model pembanding sebelum tuning. Logistic Regression dan SVM memakai fitur yang distandardisasi berdasarkan data latih. Ketiga model berbasis pohon memakai fitur tanpa standardisasi. Pembobotan kelas membuat sampel dari kelas yang lebih jarang berpengaruh lebih besar saat melatih baseline.

In [5]:
def hitung_metrik(label, prediksi):
    return {
        "Akurasi": accuracy_score(label, prediksi),
        "Macro-F1": f1_score(label, prediksi, labels=KELAS, average="macro", zero_division=0),
        "Recall Resusitasi": recall_score(label, prediksi, labels=[0], average=None, zero_division=0)[0],
    }

standardisasi = StandardScaler()
fitur_latih_standar = standardisasi.fit_transform(fitur_latih)
fitur_validasi_standar = standardisasi.transform(fitur_validasi)
bobot_seimbang = compute_sample_weight("balanced", label_latih)

model_pembanding = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED, class_weight="balanced"),
    "SVM": SVC(kernel="poly", class_weight="balanced", random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=3, random_state=SEED, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=3, random_state=SEED, class_weight="balanced"),
    "XGBoost baseline": XGBClassifier(objective="multi:softprob", num_class=3, max_depth=3, n_estimators=100, random_state=SEED),
}

hasil_validasi = {}
for nama, model in model_pembanding.items():
    print(f"Melatih {nama}...")
    if nama in ("Logistic Regression", "SVM"):
        model.fit(fitur_latih_standar, label_latih)
        prediksi = model.predict(fitur_validasi_standar)
    elif nama == "XGBoost baseline":
        model.fit(fitur_latih, label_latih, sample_weight=bobot_seimbang)
        prediksi = model.predict(fitur_validasi)
    else:
        model.fit(fitur_latih, label_latih)
        prediksi = model.predict(fitur_validasi)
    hasil_validasi[nama] = hitung_metrik(label_validasi, prediksi)

print("\nPerbandingan pada set validasi:")
print(pd.DataFrame.from_dict(hasil_validasi, orient="index").round(4).to_string())

Melatih Logistic Regression...
Melatih SVM...
Melatih Random Forest...
Melatih Decision Tree...
Melatih XGBoost baseline...

Perbandingan pada set validasi:
                     Akurasi  Macro-F1  Recall Resusitasi
Logistic Regression   0.8814    0.8908             0.9700
SVM                   0.8859    0.8943             0.9526
Random Forest         0.8814    0.8897             0.9388
Decision Tree         0.7670    0.7823             0.8812
XGBoost baseline      0.9112    0.9181             0.9706


## 6. Cari konfigurasi XGBoost dengan Optuna

Satu trial mencoba kombinasi pengaturan pohon dan bobot kelas. Pelatihan trial dapat berhenti setelah 30 putaran tanpa perbaikan `mlogloss` pada validasi. Optuna memilih skor terbesar: macro-F1 jika recall Resusitasi mencapai 95%, atau recall Resusitasi dikali 0,1 jika di bawah ambang. Ambang ini bagian dari eksperimen.

In [6]:
PARAMETER_DASAR = {
    "objective": "multi:softprob", "num_class": 3, "tree_method": "hist",
    "random_state": SEED, "n_estimators": 400,
}
AMBANG_RECALL = 0.95

def tujuan_trial(trial):
    parameter_pohon = {
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "subsample": trial.suggest_float("subsample", 0.65, 0.95),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 0.95),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 6.0),
        "gamma": trial.suggest_float("gamma", 0.0, 2.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 5.0),
    }
    bobot_kelas = {0: trial.suggest_float("w0", 1.2, 3.0), 1: 1.0, 2: trial.suggest_float("w2", 0.7, 1.5)}
    model_trial = XGBClassifier(
        **PARAMETER_DASAR, **parameter_pohon, early_stopping_rounds=30, eval_metric="mlogloss",
    )
    model_trial.fit(
        fitur_latih, label_latih,
        sample_weight=compute_sample_weight(bobot_kelas, label_latih),
        eval_set=[(fitur_validasi, label_validasi)], verbose=False,
    )
    metrik = hitung_metrik(label_validasi, model_trial.predict(fitur_validasi))
    recall = metrik["Recall Resusitasi"]
    trial.set_user_attr("macro_f1", metrik["Macro-F1"])
    trial.set_user_attr("recall_resusitasi", recall)
    if (trial.number + 1) % 10 == 0:
        print(f"Trial selesai: {trial.number + 1}/50")
    return metrik["Macro-F1"] if recall >= AMBANG_RECALL else recall * 0.1

optuna.logging.set_verbosity(optuna.logging.WARNING)
studi = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
print("Memulai 50 trial pada set validasi...")
studi.optimize(tujuan_trial, n_trials=50)
trial_terbaik = studi.best_trial
recall_terbaik = trial_terbaik.user_attrs["recall_resusitasi"]
print(f"Parameter dan bobot terbaik: {trial_terbaik.params}")
print(f"Macro-F1 validasi trial terbaik: {trial_terbaik.user_attrs['macro_f1']:.4f}")
print(f"Recall Resusitasi: {recall_terbaik:.4f} | Mencapai 95%: {'ya' if recall_terbaik >= AMBANG_RECALL else 'tidak'}")

Memulai 50 trial pada set validasi...
Trial selesai: 10/50
Trial selesai: 20/50
Trial selesai: 30/50
Trial selesai: 40/50
Trial selesai: 50/50
Parameter dan bobot terbaik: {'max_depth': 4, 'learning_rate': 0.09142060454425095, 'subsample': 0.8351893751991305, 'colsample_bytree': 0.709643736312342, 'min_child_weight': 1.3322962297920464, 'gamma': 0.2616426179572755, 'reg_alpha': 0.009075017327672704, 'reg_lambda': 4.676525581516507, 'w0': 1.8823286179428018, 'w2': 1.326625020799546}
Macro-F1 validasi trial terbaik: 0.9190
Recall Resusitasi: 0.9754 | Mencapai 95%: ya


## 7. Latih model akhir

Gunakan parameter dan bobot terbaik untuk melatih XGBoost pada data latih. Seperti tahap akhir notebook sumber, model ini memakai 400 pohon tanpa *early stopping*.

In [7]:
parameter_terbaik = trial_terbaik.params.copy()
bobot_akhir = {0: parameter_terbaik.pop("w0"), 1: 1.0, 2: parameter_terbaik.pop("w2")}
parameter_akhir = {**PARAMETER_DASAR, **parameter_terbaik}
bobot_sampel_akhir = compute_sample_weight(bobot_akhir, label_latih)
assert len(bobot_sampel_akhir) == len(fitur_latih)

model_akhir = XGBClassifier(**parameter_akhir)
model_akhir.fit(fitur_latih, label_latih, sample_weight=bobot_sampel_akhir)
print("Parameter model akhir:", parameter_akhir)
print("Bobot kelas:", bobot_akhir)

Parameter model akhir: {'objective': 'multi:softprob', 'num_class': 3, 'tree_method': 'hist', 'random_state': 42, 'n_estimators': 400, 'max_depth': 4, 'learning_rate': 0.09142060454425095, 'subsample': 0.8351893751991305, 'colsample_bytree': 0.709643736312342, 'min_child_weight': 1.3322962297920464, 'gamma': 0.2616426179572755, 'reg_alpha': 0.009075017327672704, 'reg_lambda': 4.676525581516507}
Bobot kelas: {0: 1.8823286179428018, 1: 1.0, 2: 1.326625020799546}


## 8. Baca hasil pada data uji

Bandingkan kedua XGBoost pada baris uji yang sama. Pada matriks, baris menunjukkan kelas aktual dan kolom menunjukkan prediksi. Angka diagonal adalah prediksi benar. Kolom selisih menunjukkan metrik hasil tuning dikurangi baseline; nilai negatif berarti metrik turun pada pembagian ini.

Hasil menggambarkan satu pembagian internal TriageGeist dengan rumus fitur notebook ini.

In [8]:
prediksi_baseline = model_pembanding["XGBoost baseline"].predict(fitur_uji)
prediksi_akhir = model_akhir.predict(fitur_uji)
perbandingan = pd.DataFrame({
    "Baseline": hitung_metrik(label_uji, prediksi_baseline),
    "Hasil tuning": hitung_metrik(label_uji, prediksi_akhir),
})
perbandingan["Selisih"] = perbandingan["Hasil tuning"] - perbandingan["Baseline"]

matriks = confusion_matrix(label_uji, prediksi_akhir, labels=KELAS)
assert matriks.sum() == len(label_uji)
tabel_matriks = pd.DataFrame(matriks, index=NAMA_KELAS.values(), columns=NAMA_KELAS.values())
tabel_matriks.index.name = "Aktual / Prediksi"

print("Evaluasi pada set uji:")
print(perbandingan.round(4).to_string())
print("\nConfusion matrix model akhir:")
print(tabel_matriks.to_string())

Evaluasi pada set uji:
                   Baseline  Hasil tuning  Selisih
Akurasi              0.9064        0.9048  -0.0016
Macro-F1             0.9138        0.9123  -0.0015
Recall Resusitasi    0.9766        0.9778   0.0012

Confusion matrix model akhir:
                   Resusitasi  Darurat  Non-Darurat
Aktual / Prediksi                                  
Resusitasi               1629       37            0
Darurat                    17     2002          354
Non-Darurat                 0      190         2054
